In [ ]:
!pip install -q transformers accelerate bitsandbytes langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers gradio pypdf

In [ ]:
import os
import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

# 1. NEW, verified direct URL for the Paris Agreement PDF
pdf_url = "https://unfccc.int/sites/default/files/resource/parisagreement_publication.pdf"
pdf_path = "paris_agreement.pdf"

# Add a standard browser user-agent header to avoid getting blocked or redirected to an HTML page
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("Downloading the authentic PDF framework...")
response = requests.get(pdf_url, headers=headers)

with open(pdf_path, "wb") as f:
    f.write(response.content)

# 2. Load the actual PDF
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# 3. Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

# 4. Set up Recursive Token Chunking (1000-token chunks, 200 overlap)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)
print(f"Success! Split the climate framework into {len(docs)} clean semantic chunks.")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load a strong, lightweight embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Generate embeddings and index them in a FAISS vector store
vector_store = FAISS.from_documents(docs, embedding_model)

# 3. Create a retriever that fetches the top 3 most relevant chunks (k=3)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("FAISS Semantic Indexing complete. Your vector space is ready!")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer, pipeline

model_id = "Qwen/Qwen2.5-7B-Instruct"

# 1. Configure 4-bit quantization to optimize memory usage
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 2. Load Tokenizer and Quantized Model
model_tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 3. Create a Hugging Face pipeline for text generation
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=model_tokenizer,
    max_new_tokens=512,
    temperature=0.2, # Kept low for factual, non-creative policy reporting
    do_sample=True
)
print("LLM successfully loaded into T4 GPU memory!")

In [ ]:
import gradio as gr

def rag_engine(query):
    # 1. Retrieve the top 3 most relevant context chunks from FAISS
    relevant_docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    # 2. Construct a structured system prompt for the climate policy analyst
    messages = [
        {
            "role": "system",
            "content": "You are an expert climate policy analyst. Answer the user's question accurately using ONLY the provided context framework. If the answer cannot be found in the context, politely state that you do not know."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}"
        }
    ]

    # 3. Format the prompt using the model's native chat template
    prompt = model_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 4. Generate the response
    outputs = llm_pipeline(prompt)

    # 5. Clean up the output text to extract only the assistant's reply
    generated_text = outputs[0]["generated_text"]
    response = generated_text.split("<|im_start|>assistant\n")[-1].strip()

    return response

# Build the Gradio Interface tailored to policy researchers
interface = gr.Interface(
    fn=rag_engine,
    inputs=gr.Textbox(
        lines=2,
        placeholder="e.g., What are the long-term temperature goals specified in Article 2?"
    ),
    outputs=gr.Textbox(label="Semantic Policy Analysis Summary"),
    title="🌍 Semantic Climate Policy RAG Engine",
    description="Query dense, unstructured sustainability guidelines and climate frameworks with sub-millisecond vector retrieval latency.",
    theme="soft"
)

# Launch the app with a shareable public link
interface.launch(share=True, debug=True)

Sematic Climate Policy RAG Engine & Ingestion Pipeline